# KuraCommerce: Muse Glimmer server on Kaggle

**Settings first (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**, and under Add-ons > Secrets attach **HF_TOKEN** and **SERVER_KEY**.

Run cells 1 to 6 in order. Every cell works on its own, so after a reconnect you can re-run just cells 4, 5 and 6.

In [ ]:
# CELL 1: check the GPUs and install helpers
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U huggingface_hub requests

In [ ]:
# CELL 2: find and download Muse Glimmer (safe to run again: it skips the download if the files already exist)
import glob, os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, snapshot_download

HF_SECRET_LABEL = "HF_TOKEN"     # must match the label you created under Add-ons > Secrets
QUANT = "Q4_K_M"                 # about 4-bit; if nothing matches, this cell lists the names that exist
REPO_ID_OVERRIDE = ""            # optional: paste a repo id here to skip the automatic search
MODEL_DIR = "/kaggle/working/model"
BAD_WORDS = ("lora", "mlx", "executorch", "drafter", "dflash", "bricked")
PREFERRED_OWNERS = ("meta-models", "ggml-org", "unsloth")

def on_disk():
    ggufs = sorted(glob.glob(MODEL_DIR + "/**/*.gguf", recursive=True))
    model = next((g for g in ggufs if "mmproj" not in g.lower()), None)
    mmproj = next((g for g in ggufs if "mmproj" in g.lower()), None)
    return model, mmproj

MODEL, MMPROJ = on_disk()
if MODEL:
    print("Model already downloaded. Skipping.")
else:
    api = HfApi(token=UserSecretsClient().get_secret(HF_SECRET_LABEL))

    def gguf_files(repo):
        try:
            return [f for f in api.list_repo_files(repo) if f.lower().endswith(".gguf")]
        except Exception as e:
            print("  could not read", repo, "-", type(e).__name__)
            return []

    repos = [REPO_ID_OVERRIDE] if REPO_ID_OVERRIDE else [m.id for m in api.list_models(search="Glimmer", limit=100)]
    print("Repositories found:")
    for r in repos:
        print("  ", r)

    candidates = []
    for r in repos:
        if any(w in r.lower() for w in BAD_WORDS):
            continue
        if any(QUANT.lower() in f.lower() and "mmproj" not in f.lower() for f in gguf_files(r)):
            owner = r.split("/")[0].lower()
            candidates.append((PREFERRED_OWNERS.index(owner) if owner in PREFERRED_OWNERS else 99, r))
    candidates.sort()

    if not candidates:
        print("\nNo repository has a GGUF file matching", QUANT)
        for r in repos:
            names = gguf_files(r)
            if names:
                print("  ", r, "->", names[:8])
        raise SystemExit("Copy everything printed above and send it to your helper.")

    REPO_ID = candidates[0][1]
    print("\nUsing:", REPO_ID)
    if len(candidates) > 1:
        print("Other options:", [r for _, r in candidates[1:]], "(set REPO_ID_OVERRIDE to use one)")

    files = gguf_files(REPO_ID)
    model_files = sorted(f for f in files if QUANT.lower() in f.lower() and "mmproj" not in f.lower())
    mmproj_files = sorted((f for f in files if "mmproj" in f.lower()),
                          key=lambda f: 0 if ("f16" in f.lower() or "bf16" in f.lower()) else 1)
    wanted = model_files + mmproj_files[:1]
    print("Downloading:", wanted)
    snapshot_download(REPO_ID, token=api.token, allow_patterns=wanted, local_dir=MODEL_DIR)
    MODEL, MMPROJ = on_disk()

print("MODEL: ", MODEL)
print("MMPROJ:", MMPROJ)
if not MODEL:
    raise SystemExit("Download failed: no model file found.")
if not MMPROJ:
    print("WARNING: no mmproj (vision) file. The model cannot read photos without it.")

In [ ]:
%%bash
# CELL 3: build llama.cpp with CUDA for the T4 (10-20 minutes the first time; skipped if already built)
BIN=/kaggle/working/llama.cpp/build/bin/llama-server
if [ -x "$BIN" ]; then echo "llama-server already built. Skipping."; exit 0; fi
git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
cmake --build /kaggle/working/llama.cpp/build --config Release -j 4 --target llama-server
ls -l "$BIN" && echo "BUILD OK"

In [ ]:
# CELL 4: start the model server and test it (safe to run again: it restarts the server)
import base64, glob, io, os, subprocess, time, requests
from kaggle_secrets import UserSecretsClient

SERVER_SECRET_LABEL = "SERVER_KEY"   # must match the label you created under Add-ons > Secrets
MODEL_NAME = "muse-glimmer-30b"
BIN = "/kaggle/working/llama.cpp/build/bin/llama-server"

SERVER_KEY = UserSecretsClient().get_secret(SERVER_SECRET_LABEL)
ggufs = sorted(glob.glob("/kaggle/working/model/**/*.gguf", recursive=True))
MODEL = next((g for g in ggufs if "mmproj" not in g.lower()), None)
MMPROJ = next((g for g in ggufs if "mmproj" in g.lower()), None)
if not MODEL:
    raise SystemExit("Model not downloaded. Run CELL 2 first.")
if not os.path.exists(BIN):
    raise SystemExit("llama.cpp not built. Run CELL 3 first.")

subprocess.run(["pkill", "-f", "llama-server"])
time.sleep(2)
cmd = [BIN, "-m", MODEL, "-ngl", "99", "--tensor-split", "1,1", "-c", "8192", "--host", "127.0.0.1",
       "--port", "8080", "--api-key", SERVER_KEY, "--jinja", "--alias", MODEL_NAME]
if MMPROJ:
    cmd += ["--mmproj", MMPROJ]
server = subprocess.Popen(cmd, stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)

for _ in range(120):
    time.sleep(5)
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=3).status_code == 200:
            print("Model server is ready."); break
    except Exception:
        pass
    if server.poll() is not None:
        print("Server stopped. Last log lines:"); print(open("/kaggle/working/server.log").read()[-3000:])
        raise SystemExit("Send the log above to your helper.")
else:
    raise SystemExit("Not ready after 10 minutes. See /kaggle/working/server.log")

HEAD = {"Authorization": "Bearer " + SERVER_KEY}
def ask(content, max_tokens=80):
    r = requests.post("http://127.0.0.1:8080/v1/chat/completions", headers=HEAD, timeout=180,
                      json={"model": MODEL_NAME, "messages": [{"role": "user", "content": content}], "max_tokens": max_tokens})
    return r.json()["choices"][0]["message"]["content"]

print("\nText test:", ask("Write one friendly sentence in Kinyarwanda promoting fresh tomatoes."))
try:
    from PIL import Image
    buf = io.BytesIO(); Image.new("RGB", (128, 128), (220, 20, 20)).save(buf, "JPEG")
    img = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()
    print("Vision test (should say red):", ask([{"type": "text", "text": "What colour is this image? One word."},
                                                 {"type": "image_url", "image_url": {"url": img}}], 10))
except Exception as e:
    print("Vision test FAILED:", type(e).__name__, e, "- the model may not be reading photos")

In [ ]:
# CELL 5: open a free HTTPS tunnel and print the settings for your app (run again after a restart to get a new address)
import re, subprocess, time, requests
from kaggle_secrets import UserSecretsClient

SERVER_SECRET_LABEL = "SERVER_KEY"
MODEL_NAME = "muse-glimmer-30b"
SERVER_KEY = UserSecretsClient().get_secret(SERVER_SECRET_LABEL)

subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(1)
subprocess.run("curl -sL -o /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared", shell=True)
open("/kaggle/working/tunnel.log", "w").close()
tun = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                       stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
URL = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if m:
        URL = m.group(0); break
if not URL:
    print(open("/kaggle/working/tunnel.log").read()[-2000:]); raise SystemExit("No tunnel address. Send the log above to your helper.")
print("Tunnel address:", URL)

for _ in range(8):   # a new address can take a few seconds to start working
    try:
        r = requests.get(URL + "/v1/models", headers={"Authorization": "Bearer " + SERVER_KEY}, timeout=20)
        print("Through the tunnel:", r.status_code, [x.get("id") for x in r.json().get("data", [])]); break
    except Exception as e:
        time.sleep(5)
else:
    print("Tunnel check failed. Is CELL 4 running (Model server is ready)?")

print("\nRun this on a computer to point your live app at this session (replace the 3 placeholders):")
print("curl -X POST https://YOUR-APP.onrender.com/api/admin/model-endpoint -H 'Content-Type: application/json' "
      "-d '{\"password\":\"YOUR_ADMIN_PASSWORD\",\"base_url\":\"" + URL + "/v1\",\"api_key\":\"YOUR_SERVER_KEY\"}'")
print("\nRender environment variables (set once):")
print("  META_API_BASE=" + URL + "/v1")
print("  META_API_KEY=<the same value as your SERVER_KEY secret>")
print("  META_MODEL_WONDER=" + MODEL_NAME)
print("  META_MODEL_SPARK=" + MODEL_NAME)
print("  LIVE_STEPS=wonder,spark")
print("  MODEL_PROVIDER_LABEL=Muse Glimmer (self-hosted)")
print("  TIMEOUT_SCALE=4")

In [ ]:
# CELL 6: keep the session alive while you test or demo (stop this cell to end)
import time, requests
while True:
    time.sleep(60)
    try:
        ok = requests.get("http://127.0.0.1:8080/health", timeout=5).status_code == 200
    except Exception:
        ok = False
    print("alive", time.strftime("%H:%M:%S"), "| model server", "up" if ok else "DOWN (run CELL 4)", flush=True)